In [28]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks
# Obtener la ruta del directorio raíz del proyecto
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))
# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(PROJECT_ROOT)

INPUTS_FOLDER = os.path.join(PROJECT_ROOT, 'tests', 'test_data', 'inputs')
OUTPUTS_FOLDER = os.path.join(PROJECT_ROOT, 'tests', 'test_data', 'outputs')

def get_file_from_path(file_path: str) -> str:
    return os.path.join(INPUTS_FOLDER, file_path)

test_files = [
    'banorte_credit_new.pdf',
    'banorte_credit_old.pdf',
    'banorte_debit.pdf',
    'bbva_debit.pdf',
]

file_path = get_file_from_path('bbva_debit.pdf')

In [29]:
from models import DocumentProcessingFacade
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

doc_processor = DocumentProcessingFacade(file_path)
statement_properties = doc_processor.get_statement_properties()
bank = statement_properties['bank']
statement_type = statement_properties['statement_type']
new_format = statement_properties['new_format']

print(bank, ' - ', statement_type, ' - ', new_format)

bbva  -  debit  -  None


In [30]:
corrected_extracted_words = doc_processor.get_corrected_extracted_words()
corrected_extracted_words

,page,text,x0,top,x1,bottom
0,1,Estado,510.546000,1.219000,544.314000,13.219000
1,1,de,546.858000,1.219000,559.638000,13.219000
2,1,Cuenta,562.182000,1.219000,598.182000,13.219000
3,1,Libretón,459.986607,16.654410,498.904607,27.654410
4,1,Básico,501.236607,16.654410,530.397608,27.654410
...,...,...,...,...,...,...
2006,8,de,500.295960,773.127129,508.503960,782.127129
2007,8,la,510.555960,773.127129,516.297960,782.127129
2008,8,Ley,518.349960,773.127129,530.247960,782.127129
2009,8,Personas,532.299960,773.127129,563.475960,782.127129


In [31]:
if statement_type == 'debit':
    corrected_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_corrected_words.csv')
elif statement_type == 'credit':
    _format = 'new' if new_format else 'old'
    corrected_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_corrected_words.csv')
    
corrected_extracted_words.to_csv(corrected_words_path, index=False)

In [32]:
from models import TableProcessingFacade

table_processor = TableProcessingFacade(corrected_extracted_words, statement_properties)
reconstructor = table_processor.get_reconstructor()
structured_table = reconstructor.get_structured_table()
structured_table

,Date,Description,CARGOS,ABONOS,OPERACION,LIQUIDACION
0,None,FECHA SALDO,,,,
1,None,OPER LIQ DESCRIPCION REFERENCIA CARGOS ABONOS ...,,,,
2,15/MAY,PAGO CUENTA DE TERCERO,40.00,,,
3,None,Referencia 1634723960 BNET 1592044105 PAGO PRE...,,,,
4,15/MAY,SPEI ENVIADO SANTANDER,25.00,,,
...,...,...,...,...,...,...
149,None,CETES DIRECTO,,,,
150,None,Total de Movimientos,,,,
151,None,TOTAL IMPORTE CARGOS TOTAL MOVIMIENTOS CARGOS 30,,,,
152,None,TOTAL IMPORTE ABONOS TOTAL MOVIMIENTOS ABONOS 6,,,,


In [33]:
reconstructed_table = table_processor.reconstruct_table()
reconstructed_table

,Date,Description,CARGOS,ABONOS,OPERACION,LIQUIDACION
0,15/MAY,PAGO CUENTA DE TERCERO Referencia 1634723960 ...,40.00,,,
1,15/MAY,SPEI ENVIADO SANTANDER Referencia 0059269420 ...,25.00,,,
2,15/MAY,PAGO CUENTA DE TERCERO Referencia 1641104987 ...,60.00,,"266,326.99","266,451.99"
3,17/MAY,SPEI ENVIADO SANTANDER Referencia 0063053690 ...,850.00,,"265,476.99","265,476.99"
4,18/MAY,SPEI ENVIADO STP Referencia 0064298989 646 18...,"80,000.00",,,
5,18/MAY,SPEI ENVIADO STP Referencia 0064318415 646 18...,500.00,,,
6,18/MAY,SAT Referencia GUIA:0095194 REF:042229R345003...,"13,621.00",,,
7,18/MAY,SAT Referencia GUIA:0112519 REF:042229QU05003...,"7,398.00",,,
8,18/MAY,SAT Referencia GUIA:0132561 REF:042229QD92003...,"3,832.00",,,
9,18/MAY,SAT Referencia GUIA:0146586 REF:042229Q995003...,"4,971.00",,,


In [34]:
if statement_type == 'debit':
    reconstructed_table_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_reconstructed_table.csv')
elif statement_type == 'credit':
    _format = 'new' if new_format else 'old'
    reconstructed_table_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_reconstructed_table.csv')
    
reconstructed_table.to_csv(reconstructed_table_path, index=False)

In [35]:
from models import DataProcessingFacade

data_processor = DataProcessingFacade(corrected_extracted_words, reconstructed_table, statement_properties)
df_transactions = data_processor.get_normalized_table()
df_transactions

,Date,Description,Amount,Type
0,2022-05-15,PAGO CUENTA DE TERCERO Referencia 1634723960 ...,-40.00,Cargo
2,2022-05-15,PAGO CUENTA DE TERCERO Referencia 1641104987 ...,-60.00,Cargo
36,2022-05-15,Saldo Inicial,266451.99,Saldo Inicial
1,2022-05-15,SPEI ENVIADO SANTANDER Referencia 0059269420 ...,-25.00,Cargo
3,2022-05-17,SPEI ENVIADO SANTANDER Referencia 0063053690 ...,-850.00,Cargo
4,2022-05-18,SPEI ENVIADO STP Referencia 0064298989 646 18...,-80000.00,Cargo
5,2022-05-18,SPEI ENVIADO STP Referencia 0064318415 646 18...,-500.00,Cargo
6,2022-05-18,SAT Referencia GUIA:0095194 REF:042229R345003...,-13621.00,Cargo
7,2022-05-18,SAT Referencia GUIA:0112519 REF:042229QU05003...,-7398.00,Cargo
8,2022-05-18,SAT Referencia GUIA:0132561 REF:042229QD92003...,-3832.00,Cargo


In [36]:
from models import CsvExporter

exporter = CsvExporter(df_transactions)
if statement_type == 'debit':
    normalized_table_path = os.path.join(OUTPUTS_FOLDER, f'{bank}_{statement_type}_normalized_table.csv')
elif statement_type == 'credit':
    _format = 'new' if new_format else 'old'
    normalized_table_path = os.path.join(OUTPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_normalized_table.csv')

exporter.export_data(normalized_table_path)